# Execution notebook — Preprocessing pipeline 02 (Median filter)

**Type:** execution notebook.

## Purpose

Read `02_dataset/images/`, apply optional global operations and the **Median** filter, write to `02_dataset/images_pp_2/` with suffix `_PP_PL_2`.

## Imports

Loads Category A libraries from `00_common/` via `%run`.


## Configuration

Adjust values below to produce training dataset variants (image geometry unchanged).


In [ ]:
# ==========================================================
# GLOBAL OPERATIONS CONFIGURATION
# ==========================================================

APPLY_BRIGHTNESS = False
BRIGHTNESS_OFFSET = 20

APPLY_CONTRAST = False
CONTRAST_FACTOR = 1.20

APPLY_GAMMA = False
GAMMA_VALUE = 1.10

PIPELINE_NUMBER = 2
PIPELINE_FILTER_NAME = "Median"

## Imports
 (`00_common`)

Loads functions from shared library notebooks — **without reimplementing algorithms**.

In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.image as mpimg

%run ../00_common/00_generic.ipynb
%run ../00_common/01_global_operations.ipynb
%run ../00_common/02_filtering.ipynb

## Configuration

- **Input:** `02_dataset/images`
- **Output:** `02_dataset/images_pp_2/` (consumed by segmentation)

In [ ]:
PROJECT_ROOT = find_project_root(Path.cwd().resolve())
INPUT_DIR = PROJECT_ROOT / "02_dataset" / "images"
OUTPUT_DIR = PROJECT_ROOT / "02_dataset" / f"images_pp_{PIPELINE_NUMBER}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VALID_IMAGE_EXTENSIONS = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")

print(f"Project root: {PROJECT_ROOT}")
print(f"Input: {INPUT_DIR}")
print(f"Output:  {OUTPUT_DIR}")

## Batch processing

Filtro desta pipeline: **Median**.

In [ ]:
def apply_configured_global_operations(input_image):
    processed_image = input_image
    if APPLY_CONTRAST or APPLY_BRIGHTNESS:
        alpha = CONTRAST_FACTOR if APPLY_CONTRAST else 1.0
        beta = BRIGHTNESS_OFFSET if APPLY_BRIGHTNESS else 0
        processed_image = apply_brightness_contrast(
            processed_image, alpha=alpha, b=beta
        )
    if APPLY_GAMMA:
        processed_image = apply_gamma_transform(processed_image, gamma=GAMMA_VALUE)
    return processed_image


def apply_pipeline_filter(input_image):
    return apply_median_filter(input_image)


input_files = sorted(
    p for p in INPUT_DIR.iterdir() if p.is_file() and p.suffix.lower() in VALID_IMAGE_EXTENSIONS
)

if len(input_files) == 0:
    print(f"WARNING: no images found em {INPUT_DIR}")
else:
    print(f"Processing {len(input_files)} image(s)...")

for input_path in input_files:
    original_image = load_image(input_path)
    original_height, original_width = original_image.shape

    globally_processed_image = apply_configured_global_operations(original_image)
    output_image = apply_pipeline_filter(globally_processed_image)

    output_height, output_width = output_image.shape
    if (output_height, output_width) != (original_height, original_width):
        raise ValueError(
            f"Geometry changed for {input_path.name}: "
            f"{original_height}x{original_width} -> {output_height}x{output_width}"
        )

    output_filename = f"{input_path.stem}_PP_PL_{PIPELINE_NUMBER}.png"
    output_path = OUTPUT_DIR / output_filename
    mpimg.imsave(output_path, output_image, cmap="gray")
    print(f"Guardado: {output_path.name}")

print("Pipeline 02 complete.")

## Conclusions

Pipeline 02 complete. Outputs are in `02_dataset/images_pp_2/`. Run segmentation with `DATASET_FOLDER = "images_pp_2"` when this experiment is required.
